Install Libraries


#Install Libraries

In [ ]:
!pip install fastapi nest-asyncio uvicorn

# Create Config File

In [ ]:
yaml_config =  """
               model_path: "openai/clip-vit-large-patch14"
               """
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

In [ ]:
from PIL import Image
import requests

from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(text=["a photo of a cat", "a photo of a dog"], images=image, return_tensors="pt", padding=True)

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image # this is the image-text similarity score
probs = logits_per_image.softmax(dim=1) # we can take the softmax to get the label probabilities
print(probs)


# Build Model

In [ ]:
from PIL import Image
import requests

from transformers import CLIPProcessor, CLIPModel
from omegaconf import OmegaConf

class ZeroShotClassifier:
  """
  Thực hiện Zero - shot CLIP
  """
  labels = [
      "a photo of cat",
      "a photo of dog",
      "a photo of chicken",
  ]
  def __init__(self, config_path : str) -> None:
    """
    Khởi tạo mô hình Zero-shot
    Args:
      config_path: Đường dẫn đến file cấu hình

    Returns:
      None
    """
    self.config = OmegaConf.load(config_path)
    self.model = CLIPModel.from_pretrained(self.config.model_path)
    self.processor = CLIPProcessor.from_pretrained(self.config.model_path)

  def __call__(self, image : Image.Image, k : int) -> list:
    """
    Thực hiện Zero-shot trên một ảnh
    Args:
      image : Ảnh cần dự đoán
      k : top k nhãn chính xác nhất

    Returns:
      list: Danh sách các top k dictionary chứa nhãn và độ chính xác
    """

    inputs = self.processor(text = self.labels,images = image, return_tensors = "pt", padding = True)
    outputs = self.model(**inputs)

    logits_per_image = outputs.logits_per_image # this is the image-text similarity score
    probs = logits_per_image.softmax(dim=1) # we can take the softmax to get the label probabilities

    top_probs, top_labels = probs.topk(k, dim=1)
    # Ép kiểu tensor về list
    top_probs_list = top_probs.detach().cpu().tolist()[0]
    top_labels_list = top_labels.detach().cpu().tolist()[0]

    results = []
    for prob, index in zip(top_probs_list, top_labels_list):
      label_name = self.labels[index]
      results.append({
            "label": label_name,

            "confidence": round(prob * 100, 2)
        })
    return results


# Initialize Model

In [ ]:
classifier = ZeroShotClassifier("./config.yaml")

In [ ]:
url = "https://images.pexels.com/photos/13663678/pexels-photo-13663678.jpeg"
res = classifier(url, 3)
print(res)

# Initialize API

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import requests
from io import BytesIO
import nest_asyncio
import uvicorn
import threading
import asyncio
import subprocess
import re
import time
import sys

nest_asyncio.apply()
#Khởi tạo FastAPI
app = FastAPI(
    title="Zero-shot CLIP API",
    description="API phân loại ảnh zero-shot bằng mô hình CLIP",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

@app.get("/")
def root():
    return {
        "message": "Zero-shot CLIP API đang hoạt động!",
        "endpoints": {
            "predict_url": "GET /predict?image_url=<url>&k=3",
            "health": "GET /health",
            "labels": "GET /labels"
        }
    }

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/labels")
def get_labels():
    return {"labels": classifier.labels}

# def process(path_image: str, k: int):
#     """Tải ảnh + chạy inference — toàn bộ trong thread riêng"""
#     image = Image.open(requests.get(path_image, stream=True).raw)
#     return zero_shot(image, k)

def _run_inference(image_url: str, k: int) -> list:
    """Hàm inference chạy trong thread riêng."""
    try:
        response = requests.get(image_url, stream=True, timeout=30)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")
        return classifier(image, k)
    except requests.exceptions.RequestException as e:
        raise ValueError(f"Không thể tải ảnh từ URL: {e}")

# @app.get('/predict')
# async def predict(path_image: str, k: int = 3):
#     try:
#         loop = asyncio.get_event_loop()
#         # Đưa cả tải ảnh lẫn inference ra thread riêng
#         res = await loop.run_in_executor(None, process, path_image, k)
#         return res
#     except Exception as e:
#         return {"error": str(e)}

@app.get("/predict")
async def predict(image_url: str, k: int = 3):
    """
    Dự đoán nhãn cho ảnh từ URL.

    - **image_url**: URL của ảnh cần phân loại
    - **k**: Số lượng nhãn top-k trả về (mặc định: 3)
    """
    if not image_url:
        raise HTTPException(
            status_code=400, detail="image_url không được để trống")
    if k < 1:
        raise HTTPException(status_code=400, detail="k phải >= 1")

    try:
        loop = asyncio.get_event_loop()
        results = await loop.run_in_executor(None, _run_inference, image_url, k)
        return {"image_url": image_url, "k": k, "results": results}
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Lỗi server: {str(e)}")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)
print("Server started on http://0.0.0.0:8000")

# Call Local API

In [ ]:
import requests
import threading

result = {}

def call_api():
    API_URL = "http://127.0.0.1:8000/predict"
    params = {
        "image_url": "https://images.pexels.com/photos/13663678/pexels-photo-13663678.jpeg",
        "k": 3
    }
    response = requests.get(API_URL, params=params, timeout=120)
    result['data'] = response.json()

t = threading.Thread(target=call_api)
t.start()
t.join(timeout=120)

print(result.get('data', 'Timeout hoặc lỗi'))

# Get Pinggy URL

In [ ]:
import subprocess, threading, time, re

public_url = None

def start_pinggy():
    global public_url
    process = subprocess.Popen(
        ["ssh", "-p", "443", "-R", "0:localhost:8000",
         "-o", "StrictHostKeyChecking=no",
         "-o", "ServerAliveInterval=30", "a.pinggy.io"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in process.stdout:
        print(line, end="")
        # bắt .pinggy.link lẫn .pinggy-free.link
        match = re.search(r'https://\S+\.pinggy[\w-]*\.link', line)
        if match:
            public_url = match.group(0)
            print(f"\n Public URL: {public_url}/predict")
            break

threading.Thread(target=start_pinggy, daemon=True).start()
time.sleep(5)
print(f"Dùng URL: {public_url}/predict")

# Call Public API

In [ ]:
import requests
import threading

result = {}

def call_api():
    API_URL = public_url + "/predict"
    params = {
        "image_url": "https://images.pexels.com/photos/13663678/pexels-photo-13663678.jpeg",
        "k": 3
    }
    response = requests.get(API_URL, params=params, timeout=120)
    result['data'] = response.json()

t = threading.Thread(target=call_api)
t.start()
t.join(timeout=120)

print(result.get('data', 'Timeout hoặc lỗi'))